# CWS Model Test Notebook

Loads the fixed CWS pilot model, summarizes graph content, runs JSON Schema
and semantic validation via `KGModelValidator`, and displays the findings.

**Paths** are resolved relative to this notebook's directory, so the notebook
can be run from any working directory.

## 1. Setup paths

In [ ]:
from pathlib import Path
import json
import importlib.util
import sys
import pandas as pd

# Resolve notebook directory regardless of Jupyter launch location
try:
    NOTEBOOK_DIR = Path(__file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()  # fallback: cwd (run notebook from test_mbse/)

VALIDATOR_DIR = (NOTEBOOK_DIR / '../../../knowledge_graph/validator').resolve()
SCHEMA_DIR    = (NOTEBOOK_DIR / '../../../knowledge_graph/schemas').resolve()

MODEL_PATH     = NOTEBOOK_DIR / 'model' / 'cws_pilot_model.json'
VALIDATOR_PATH = VALIDATOR_DIR / 'kg_model_validator.py'
TOML_SCHEMAS   = [
    SCHEMA_DIR / 'mbseSchema.toml',
]
REPORT_OUT = NOTEBOOK_DIR / 'model' / 'validation_report.json'

for p, label in [(MODEL_PATH, 'model'), (VALIDATOR_PATH, 'validator')] + [(p, p.name) for p in TOML_SCHEMAS]:
    status = 'OK' if p.exists() else 'MISSING'
    print(f'[{status}] {label}: {p}')

## 2. Load model and schema

In [ ]:
with MODEL_PATH.open('r', encoding='utf-8') as f:
    model = json.load(f)

model['metadata']

## 3. Graph content summary

In [ ]:
summary = {f'nodes.{k}': len(v) for k, v in model.get('nodes', {}).items()}
summary.update({f'relations.{k}': len(v) for k, v in model.get('relations', {}).items()})
(
    pd.DataFrame(summary.items(), columns=['collection', 'count'])
    .sort_values('collection')
    .reset_index(drop=True)
)

## 4. Load validator and run validation

In [ ]:
# Dynamically load kg_model_validator from the knowledge_graph/validator package
_spec = importlib.util.spec_from_file_location('kg_model_validator', VALIDATOR_PATH)
_mod  = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = _mod
_spec.loader.exec_module(_mod)

validator = _mod.KGModelValidator(model, toml_schema_paths=TOML_SCHEMAS)
report    = validator.validate()

report['summary']

## 5. Findings table

In [ ]:
findings = pd.DataFrame(report['findings'])
findings

## 6. Findings breakdown by severity and rule

In [ ]:
if findings.empty:
    print('No findings — model is clean.')
else:
    (
        findings.groupby(['severity', 'rule_id'])
        .size()
        .reset_index(name='count')
        .sort_values(['severity', 'count'], ascending=[True, False])
    )

## 7. Connector-port detail (C1/C5 diagnostic)

In [ ]:
cp = pd.DataFrame(model['relations'].get('connects_port', []))
cn = pd.DataFrame(model['nodes'].get('connector', []))[['id', 'name', 'directionality']]

if cp.empty or cn.empty:
    print('No connector/port data found.')
else:
    (
        cp.merge(cn, left_on='from', right_on='id', how='left')
        [['from', 'name', 'to', 'endpoint_role', 'direction_hint', 'sequence_index']]
        .sort_values(['from', 'sequence_index'])
    )

## 8. Save report to disk

In [ ]:
REPORT_OUT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(f'Report written to: {REPORT_OUT}')

## 9. Visualize CWS

In [ ]:
from graphviz import Digraph

# BDD
bdd = Digraph()
bdd.attr(rankdir='TB')

bdd.node('CWS')

for s in ['PumpTrainA','PumpTrainB','PumpTrainC',
          'ScreenA','ScreenB','ScreenC',
          'CondenserA','CondenserB','CondenserC',
          'Monitoring']:
    bdd.node(s)
    bdd.edge('CWS', s)

bdd.node('Pump')
bdd.node('Motor')
bdd.edge('PumpTrainA','Pump')
bdd.edge('Pump','Motor')

bdd

In [ ]:
# IBD
ibd = Digraph()
ibd.attr(rankdir='LR')

ibd.node('ScreenA')
ibd.node('PumpA')
ibd.node('ValveA')
ibd.node('CondenserA')
ibd.node('Monitoring')

ibd.edge('ScreenA','PumpA', label='Water')
ibd.edge('PumpA','ValveA', label='Water')
ibd.edge('ValveA','CondenserA', label='Water')

ibd.node('FlowTx')
ibd.node('PressureTx')

ibd.edge('PumpA','FlowTx', label='sense')
ibd.edge('PumpA','PressureTx', label='sense')
ibd.edge('FlowTx','Monitoring', label='flow signal')
ibd.edge('PressureTx','Monitoring', label='pressure signal')

ibd